In [1]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [2]:
def create_mfcc_features(df, clean_dir, n_mfcc=13, hop_length=512, sr=16000):
    clean_dir = Path(clean_dir)
    mfcc_features_list = []
    
    for f in tqdm(df.fname):
        y, _ = librosa.load(clean_dir / f, sr=sr)
        n_fft = min(2048, len(y))
        mfcc = librosa.feature.mfcc(y=y, 
                                    sr=sr, 
                                    n_mfcc=n_mfcc, 
                                    n_fft=n_fft, 
                                    hop_length=hop_length)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        mfcc_features = np.concatenate([mfcc_mean, mfcc_std])
        mfcc_features_list.append(mfcc_features)

    X = np.array(mfcc_features_list)
    y = df['label'].values

    return X, y

Load train/test DataFrame

In [3]:
train_csv_path = 'kaggle_meta/train_post_competition.csv'
train_df = pd.read_csv(train_csv_path)

In [4]:
# musical_instruments = [
#     'Hi-hat', 'Saxophone', 'Trumpet', 'Glockenspiel', 'Cello', 'Knock',
#     'Gunshot_or_gunfire', 'Clarinet', 'Computer_keyboard',
#     'Keys_jangling', 'Snare_drum', 'Writing', 'Laughter', 'Tearing',
#     'Fart', 'Oboe', 'Flute', 'Cough', 'Telephone', 'Bark', 'Chime',
#     'Bass_drum', 'Bus', 'Squeak', 'Scissors', 'Harmonica', 'Gong',
#     'Microwave_oven', 'Burping_or_eructation', 'Double_bass',
#     'Shatter', 'Fireworks', 'Tambourine', 'Cowbell', 'Electric_piano',
#     'Meow', 'Drawer_open_or_close', 'Applause', 'Acoustic_guitar',
#     'Violin_or_fiddle', 'Finger_snapping'
# ]

musical_instruments = [
    'Hi-hat', 'Saxophone', 'Trumpet', 'Glockenspiel', 'Cello', 
    'Clarinet', 'Oboe', 'Flute', 'Bass_drum', 'Double_bass',
    'Tambourine', 'Cowbell', 'Electric_piano', 'Harmonica', 
    'Acoustic_guitar', 'Violin_or_fiddle'
]

train_df = train_df[train_df['label'].isin(musical_instruments)]

In [5]:
train_df = train_df[train_df['fname'].apply(lambda x: os.path.exists(os.path.join('kaggle_data', x)))]
train_df = train_df[train_df['manually_verified'] == 1]
train_df = train_df.reset_index(drop=True)

Load data

In [6]:
X, y = create_mfcc_features(train_df, 'kaggle_clean')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

100%|██████████| 1847/1847 [02:29<00:00, 12.38it/s]


Train SVM

In [ ]:
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='linear',
    max_iter=-1, probability=False, random_state=None, shrinking=True,
    tol=0.001, verbose=False)

Evaluate SVM

In [ ]:
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print('Accuracy:', accuracy)
print('\nClassification Report:\n', class_report)

# output_file = 'svm_evaluation.txt'

# with open(output_file, 'w') as file:
#     file.write(f'Accuracy: {accuracy}\n')
#     file.write('\nClassification Report:\n')
#     file.write(class_report)

# best .76

Accuracy: 0.7621621621621621

Classification Report:
                   precision    recall  f1-score   support

 Acoustic_guitar       0.64      0.67      0.65        21
       Bass_drum       0.85      0.85      0.85        13
           Cello       0.92      0.92      0.92        25
        Clarinet       0.67      0.46      0.55        26
         Cowbell       0.85      0.89      0.87        19
     Double_bass       0.78      1.00      0.88        18
  Electric_piano       0.83      0.67      0.74        15
           Flute       0.53      0.73      0.61        26
    Glockenspiel       0.82      1.00      0.90        14
       Harmonica       0.67      0.56      0.61        18
          Hi-hat       0.89      0.89      0.89        18
            Oboe       0.68      0.85      0.76        20
       Saxophone       0.70      0.73      0.71        51
      Tambourine       0.90      0.95      0.92        19
         Trumpet       1.00      0.65      0.79        17
Violin_or_fiddle 

Save SVM

In [9]:
# model_filepath = 'svm_model.pkl'

# with open(model_filepath, 'wb') as model_file:
#     pickle.dump(svm, model_file)

# print(f'SVM saved to {os.path.abspath(output_file)}')